In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sqlalchemy import create_engine

engine = create_engine("postgresql://berfinkilinc@localhost/pos_db")

In [ ]:
query= "SELECT * FROM pos_terminal_features"

df = pd.read_sql(query, engine)
print(f'veri çekildi : {len(df)}')

In [ ]:
correlations = df.corr(numeric_only=True)["risk_flag"].sort_values(key=abs, ascending=False)
print(correlations)

In [ ]:
if "risk_flag" in df.columns:

    leakage_cols = [
        "risk_flag", "terminal_id", "sector", "recent_device_fault", "sector_deviation"
    ]
    X = df.drop(columns= leakage_cols, errors= "ignore")
    y= df["risk_flag"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print("tamamdır")
else:
    print("risk_flag bulunamadı, sütun eksik.")

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score


In [ ]:
mlflow.set_experiment("POS_Arıza_Risk_Tahmini")

with mlflow.start_run():
    n_estimators = 100 #r andom foresttaki ağaç sayısı
    max_depth = 5 # leaf derinliği (çok derin olursa sıkıntı)

    # parametleri mflow a logluyoruz
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # model kurulumu
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # tahminler
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # metrikler

    roc_auc = roc_auc_score(y_test,y_prob)

    #mflow a loglanması

    mlflow.log_metric("roc_auc", roc_auc)

    print(f"model eğitildi. roc-auc scoru: {roc_auc:.4f}")
    print("\nsinfilandirma rapuro:")
    print(classification_report(y_test,y_pred))

    # modeli mlflow a kaydet

    mlflow.sklearn.log_model(model, "random_forest_model")
